# Econ 390 - Problem Set 10 Answers
## by M McMain
This problem set will cover Merging. Be sure to import any packages your code needs to run and make sure the code runs without any errors.

1. [1 Point] Read in the raw Federal Funds Rate data that I have uploaded [here](https://m-mcmain.github.io/files/Econ390SP26/FEDFUNDS.csv). Save it as `ffr_cleaned`.
   - To get it into merge-ready form, create the variable `Year` by subsetting  `observation_date`. *Hint: `observation_date` is read in as a string, how do you subset a string? Which elements of it do you need?*
   - Convert the `Year` to an integer
   - Since `observation_date` is no longer useful, drop it from the DataFrame
   - Set `Year` as index if you haven't already, that way you can merge on index later on

In [1]:
import pandas as pd

In [2]:
ffr_cleaned = pd.read_csv("https://m-mcmain.github.io/files/Econ390SP26/FEDFUNDS.csv")
ffr_cleaned["Year"] = ffr_cleaned["observation_date"].str[0:4].astype("int")
ffr_cleaned = ffr_cleaned.drop(["observation_date"], axis=1)
ffr_cleaned = ffr_cleaned.set_index("Year")

2. [2 Points] Read in the raw BEA data that I have uploaded [here](https://m-mcmain.github.io/files/Econ390SP26/rd_BEA.xlsx). Save it as `rd_BEA`.
   - Update `rd_BEA` to include only observations where `LineCode` is equal to 1 `TableName` is equal to SARDCOMP.
   - Keep only the variables `GeoName` and the years (`2012`-`2023`)
   - Rename `GeoName` to `State`
   - Use the `.melt` method to convert the data from Wide to Tall. Use the `var_name` option to create `Year` and `value_name` option to create `RD`
   - Set `Year` as index if you haven't already, that way you can merge on index later on

In [3]:
# A lot of this could be done in fewer lines, or even one, but I'm splitting it up to match up with the bullets above
rd_BEA = pd.read_excel("https://m-mcmain.github.io/files/Econ390SP26/rd_BEA.xlsx")
rd_BEA = rd_BEA[(rd_BEA["LineCode"] == 1.0) & (rd_BEA["TableName"] == "SARDCOMP")]
rd_BEA = rd_BEA.drop(["GeoFIPS", "TableName", "LineCode", "Description", "Unit"], axis = 1)
rd_BEA = rd_BEA.rename(columns={"GeoName":"State"})
rd_BEA = pd.melt(rd_BEA, id_vars="State", var_name = "Year", value_name = "RD")
rd_BEA = rd_BEA.set_index("Year")

3. [1 Point] Merge `ffr_cleaned` and `rd_BEA` keeping only the rows that are successfully matched.

In [4]:
rd_merged = pd.merge(ffr_cleaned, rd_BEA, left_index = True, right_index = True, indicator = True)
rd_merged

,FEDFUNDS,State,RD,_merge
Year,,,,
2012,0.14,United States,262831.4,both
2012,0.14,Alabama,2176.2,both
2012,0.14,Alaska,186.1,both
2012,0.14,Arizona,4265.9,both
2012,0.14,Arkansas,439.1,both
...,...,...,...,...
2023,5.02,Virginia,10372,both
2023,5.02,Washington,42498.5,both
2023,5.02,West Virginia,550.4,both


4. [2 Points] Now, remerge the data to look at the columns that were not matched when merging `ffr_cleaned` and `rd_BEA`
   - Print out all rows that are only in `ffr_cleaned`. *Hint: The `indicator` option will make this very easy*
   - Print out all rows that are only in `rd_BEA`
   - Reflect on if this is a problem or not

In [5]:
rd_merged = pd.merge(ffr_cleaned, rd_BEA, left_index = True, right_index = True, how="outer", indicator=True)
print(rd_merged[rd_merged["_merge"] == "left_only"])
print(rd_merged[rd_merged["_merge"] == "right_only"])

      FEDFUNDS State   RD     _merge
Year                                
2008      1.93   NaN  NaN  left_only
2009      0.16   NaN  NaN  left_only
2010      0.18   NaN  NaN  left_only
2011      0.10   NaN  NaN  left_only
2024      5.14   NaN  NaN  left_only
Empty DataFrame
Columns: [FEDFUNDS, State, RD, _merge]
Index: []


Reflection: This is not a problem! We have a limited amount of data on R&D, but we have FFR data for every year we have R&D data, so from 2012-2023 we have a "complete" dataset.